In [11]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import IFrame, display

root = Path('/home/chenwu/Documents/Github/ViMoR')
default_df = pd.read_csv(root / 'outputs/power_monitor/default_dvfs_workloads.csv')
full_df = pd.read_csv(root / 'outputs/power_monitor/full_dvfs_workloads.csv')

for c in ['workload_config', 'workload_apps', 'cpu_freq_target_mhz', 'gpu_freq_target_mhz']:
    default_df[c] = default_df[c].astype(str)
    full_df[c] = full_df[c].astype(str)

rows = []
for wid in sorted(default_df['workload_config'].unique()):
    d = default_df[
        (default_df['workload_config'] == wid)
        & (default_df['cpu_freq_target_mhz'] == 'default')
        & (default_df['gpu_freq_target_mhz'] == 'default')
    ]
    if d.empty:
        continue
    d = d.iloc[0]
    apps = str(d['workload_apps'])
    if 'llm' not in apps:
        continue

    default_t_llm = float(d['llm_e2e_s_mean'])
    default_p_mean = float(d['mean_power_w'])
    default_energy = default_p_mean * default_t_llm

    candidates = []
    for _, r in full_df[full_df['workload_config'] == wid].iterrows():
        if pd.isna(r.get('llm_e2e_s_mean')) or pd.isna(r.get('mean_power_w')):
            continue
        cand_t_llm = float(r['llm_e2e_s_mean'])
        cand_p_mean = float(r['mean_power_w'])
        cand_energy = cand_p_mean * cand_t_llm

        # LLM latency should not regress beyond 0.5%; use the average end-to-end latency as the
        # single-request completion time estimate, and compute energy as P_mean * T_LLM.
        perf_ratio = default_t_llm / cand_t_llm if cand_t_llm > 0 else 1.0
        if perf_ratio >= 0.995 and cand_energy < default_energy:
            candidates.append({
                'workload': wid,
                'cpu': str(r['cpu_freq_target_mhz']),
                'gpu': str(r['gpu_freq_target_mhz']),
                'energy_j': cand_energy,
                'mean_power_w': cand_p_mean,
                'perf_ratio': perf_ratio,
            })
    if not candidates:
        continue

    best = min(candidates, key=lambda x: x['energy_j'])
    rows.append({
        'workload': wid,
        'default_energy_j': default_energy,
        'default_power_w': default_p_mean,
        'matched_energy_j': float(best['energy_j']),
        'matched_power_w': float(best['mean_power_w']),
        'matched_cpu': best['cpu'],
        'matched_gpu': best['gpu'],
        'perf_ratio': best['perf_ratio'],
        'saving_pct': (1 - best['energy_j'] / default_energy) * 100,
    })

comparison = pd.DataFrame(rows).sort_values('workload').reset_index(drop=True)
print(comparison[['workload', 'default_energy_j', 'matched_energy_j', 'matched_cpu', 'matched_gpu', 'saving_pct']].to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(comparison))
width = 0.35
ax.bar(x - width/2, comparison['default_energy_j'], width=width, label='Default DVFS', color='#5B8FF9')
ax.bar(x + width/2, comparison['matched_energy_j'], width=width, label='Perf-matched lower-energy', color='#F6BD16')

for i, row in comparison.iterrows():
    ax.text(i - width/2, row['default_energy_j'] + 10, f'{row["default_energy_j"]:.0f} J', ha='center', va='bottom', fontsize=9)
    ax.text(i + width/2, row['matched_energy_j'] + 10, f'{row["matched_energy_j"]:.0f} J', ha='center', va='bottom', fontsize=9)
    ax.text(i, max(row['default_energy_j'], row['matched_energy_j']) * 0.96, f'{row["saving_pct"]:.1f}%\\nless', ha='center', va='top', fontsize=9, color='darkgreen')

ax.set_xticks(x)
ax.set_xticklabels(comparison['workload'])
ax.set_ylabel('Estimated energy (J)')
ax.set_title('Default DVFS vs. LLM request energy (P_mean × T_LLM)')
fig.tight_layout()
out_path = root / 'figures' / 'default_vs_llm_request_energy_bar.pdf'
fig.savefig(out_path, dpi=200, bbox_inches='tight')
print(f'\nSaved: {out_path}')
display(IFrame(src=str(out_path), width='100%', height=600))

workload  default_energy_j  matched_energy_j matched_cpu matched_gpu  saving_pct
      W3         82.799964         71.749907        1497         918   13.345486
      W5        102.505880         89.326309        1497         918   12.857380
      W6        130.778717        114.817135        1651         918   12.205030
      W7        166.560424        151.563601        1651         918    9.003834

Saved: /home/chenwu/Documents/Github/ViMoR/figures/default_vs_llm_request_energy_bar.pdf


# 只比较带 LLM 的 workload

这里的对比标准是：
- `default`：默认动态 DVFS 运行点
- `perf_matched`：仅在带 LLM 的 workload 中，从固定 DVFS 网格里筛出“LLM latency 不劣于 default（至少 99.5%）且能耗更低”的最优点
- 结果用柱状图展示每个带 LLM workload 的 `energy_j` 对比
